# Variance Analysis: Why One Evaluation Run Is Not Enough

Based on: [On Randomness in Agentic Evals](https://arxiv.org/abs/2602.07150) (Feb 2026)

## The Problem

You run an evaluation and get a score of 0.75. Is that reliable?

The "On Randomness" paper analyzed **60,000 agent evaluation runs** and found:
- The **same agent, same task, same evaluator** produces different scores across runs
- A single run can be off by ±0.15 from the true mean
- Pass/fail decisions based on a single run have a **20-30% error rate**

## The Technique: Multiple Runs + Statistical Reporting

Instead of one run, do 5. Report mean, standard deviation, and confidence:

```
Single run:   0.75       ← Is this reliable? No way to know.
5 runs:       0.80, 0.75, 0.70, 0.85, 0.75
              Mean: 0.77  Stdev: 0.05  ← Reliable! Low variance.

5 runs:       0.90, 0.40, 0.75, 0.60, 0.85
              Mean: 0.70  Stdev: 0.19  ← Unreliable! High variance.
```

## What We Compare

| Approach | Runs | Reliability | Cost |
|----------|:----:|:-----------:|:----:|
| Single run | 1 | ❌ Unknown | 1x |
| 5 runs + stats | 5 | ✅ Measured | 5x |
| 5 runs + 3 models | 15 | ✅✅ High | 15x |

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Test 1: Single Run vs 5 Runs

**What this does:** Evaluates the exact same response 5 times with the same evaluator and rubric, then reports the statistical spread.

**Why 5 runs:** A single evaluation run gives you one number — say, 0.75. But is that the "true" score, or would you get 0.60 or 0.90 on a different run? By running 5 times, you can compute the **standard deviation (stdev)**, which tells you how much scores vary in practice.

**What standard deviation means in practical terms:**
- **Stdev < 0.05:** Scores are stable. A single run is reliable enough for most decisions.
- **Stdev 0.05-0.15:** Moderate variance. Use the mean of 3+ runs for decisions like deployment gates.
- **Stdev > 0.15:** High variance. The evaluator or rubric is unreliable for this response — do not trust any single run. Either improve the rubric or accept that this response is genuinely ambiguous.

For context: if the mean is 0.77 with stdev 0.05, the true score is likely between 0.72 and 0.82. If the mean is 0.70 with stdev 0.19, the true score could be anywhere from 0.51 to 0.89 — far too wide for a pass/fail decision at a 0.70 threshold.

> **What to look for:** The 5 scores should cluster near the same value. If they spread widely (e.g., range from 0.40 to 0.90), that signals the rubric is ambiguous or the response is borderline. The stdev value is the key number — below 0.10 is acceptable for most use cases.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

import statistics
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

MODEL = "gpt-4o-mini"

# A borderline response (some good info, some embellishment)
RESPONSE = (
    "Flights from NYC to London:\n"
    "1. BA117 - JFK 7PM to LHR 7AM - $450\n"
    "2. DL1 - JFK 9:30PM to LHR 9:30AM - $520\n"
    "Both are excellent choices with great in-flight service!"
)

RUBRIC = (
    "Rate 0-1. 0.8+: Specific flights with details. "
    "0.5-0.7: Good but with embellishments. "
    "0.0-0.4: Vague or fabricated."
)

evaluator = OutputEvaluator(rubric=RUBRIC, model=MODEL)
case = Case(name="borderline", input="Find flights NYC to London", expected_output="Flight details")

NUM_RUNS = 5
scores = []

print("=" * 60)
print(f"TEST 1: {NUM_RUNS} RUNS of the same evaluation")
print("=" * 60)

for i in range(NUM_RUNS):
    exp = Experiment(cases=[case], evaluators=[evaluator])
    reports = exp.run_evaluations(lambda c: RESPONSE)
    score = reports[0].overall_score
    scores.append(score)
    print(f"  Run {i+1}: {score:.2f}")

print(f"\n📊 Statistics:")
print(f"   Mean:   {statistics.mean(scores):.3f}")
print(f"   Median: {statistics.median(scores):.3f}")
print(f"   Stdev:  {statistics.stdev(scores):.3f}")
print(f"   Range:  {min(scores):.2f} - {max(scores):.2f}")
print(f"   Spread: {max(scores) - min(scores):.2f}")

if statistics.stdev(scores) > 0.1:
    print(f"\n  ⚠️  High variance! A single run would be unreliable.")
else:
    print(f"\n  ✅ Low variance. Scores are stable.")

---
## Test 2: Compare variance across response quality levels

**What this does:** Runs the same evaluation 3 times each on three responses of different quality: clearly good, borderline, and clearly bad.

**The hypothesis:** Borderline responses should produce **higher variance** than clearly good or clearly bad responses. This is because LLM judges agree when the answer is obviously right or obviously wrong, but disagree when the answer is in the gray zone.

**Why this matters:** If your agent mostly produces borderline responses (which is common in real applications), then single-run evaluations are particularly unreliable. The responses where you most need accurate scores are exactly the ones where scores vary the most. This is why the "On Randomness" paper recommends a minimum of 5 independent runs per task.

| Response Quality | Expected Variance | Why |
|-----------------|-------------------|-----|
| Clearly good (specific flights with details) | Low — judges agree it is good | Clear match to rubric criteria |
| Borderline (some details, some embellishment) | High — judges disagree | Ambiguous: does embellishment count as "good info"? |
| Clearly bad (vague, no details) | Low — judges agree it is bad | Clear mismatch from rubric criteria |

> **What to look for:** The stdev for the "borderline" response should be noticeably higher than for "clearly_good" and "clearly_bad." If all three have similar variance, the rubric may be unusually clear (good) or the model may be unusually consistent.

In [ ]:
RESPONSES = {
    "clearly_good": (
        "Flights from NYC to London:\n"
        "1. BA117 - JFK 7PM to LHR 7AM - $450\n"
        "2. DL1 - JFK 9:30PM to LHR 9:30AM - $520"
    ),
    "borderline": (
        "Flights from NYC to London:\n"
        "1. BA117 - $450 (great in-flight service!)\n"
        "2. DL1 - $520"
    ),
    "clearly_bad": "There are some flights available. Check the airline websites.",
}

print("=" * 60)
print("TEST 2: VARIANCE BY RESPONSE QUALITY (3 runs each)")
print("=" * 60)

NUM_RUNS = 3
all_results = {}

for label, response in RESPONSES.items():
    run_scores = []
    for _ in range(NUM_RUNS):
        exp = Experiment(cases=[case], evaluators=[evaluator])
        reports = exp.run_evaluations(lambda c, r=response: r)
        run_scores.append(reports[0].overall_score)

    mean = statistics.mean(run_scores)
    stdev = statistics.stdev(run_scores) if len(run_scores) > 1 else 0
    all_results[label] = {"scores": run_scores, "mean": mean, "stdev": stdev}

    stability = "🟢 Stable" if stdev < 0.05 else "🟡 Moderate" if stdev < 0.15 else "🔴 Unstable"
    print(f"\n  {label}:")
    print(f"    Scores: {[f'{s:.2f}' for s in run_scores]}")
    print(f"    Mean: {mean:.3f}  Stdev: {stdev:.3f}  {stability}")

print(f"""
  ┌─────────────────────────────────────────────────────────┐
  │ Quality Level    │ Expected Variance                    │
  ├──────────────────┼──────────────────────────────────────┤
  │ Clearly good     │ 🟢 Low (judges agree)                │
  │ Borderline       │ 🔴 High (judges disagree)            │
  │ Clearly bad      │ 🟢 Low (judges agree)                │
  └──────────────────┴──────────────────────────────────────┘

  💡 High variance = the rubric needs work, or the response
     is genuinely ambiguous. Either way, a single run is not
     enough to make a decision.
""")

## Key Takeaways

1. **A single evaluation run is statistically insufficient.** The same evaluation can vary by ±0.15 across runs. Always run at least 3-5 times for decisions that matter (deployment gates, regression tests).

2. **Borderline responses have the highest variance.** Clearly good and clearly bad responses are stable. The hard cases are where you need multiple runs the most.

3. **Report mean + stdev, not a single number.** "Score: 0.77 ± 0.05" is far more useful than "Score: 0.75".

4. **High variance is a signal.** If stdev > 0.15, either: (a) your rubric is ambiguous, or (b) the response is genuinely borderline. Both are actionable insights.

5. **Cost vs reliability tradeoff.** 5 runs cost 5x more. Use multiple runs for high-stakes decisions; single runs for development iteration.

### Recommendation from the Paper

> "We recommend a minimum of 5 independent runs per task and reporting pass@k metrics instead of single-run pass/fail."

**Series complete!** You now know how to evaluate trajectories (Demo 01), compute risk scores (Demo 02), and handle evaluation variance (Demo 03).